# Loan/Card Default Prediction with Information-Theoretic ML

This notebook builds a complete loan or credit-card default prediction workflow that explicitly uses Information Theory in several stages: feature selection (Mutual Information), borrower risk quantification (Shannon Entropy), compressed representations (Information Bottleneck), and explainable modeling (Information Gain in decision trees).


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(x=tree_importances.head(8), y=tree_importances.head(8).index, palette='magma')
plt.title('Information Gain-based Feature Importance (Decision Tree)')
plt.xlabel('Normalized Information Gain')
plt.ylabel('Feature')
plt.show()

plt.figure(figsize=(16, 6))
plot_tree(tree, feature_names=selected_feature_names, class_names=['No Default', 'Default'], filled=True, rounded=True, max_depth=3)
plt.title('Entropy-Driven Decision Tree (Top Levels)')
plt.show()


## 7.1 Model Performance Analysis

Based on the results:

**Best Model: InfoBottleneck-LogReg**
- **ROC AUC: 0.7488** (highest)
- **Accuracy: 0.8845** (highest)
- This model combines the Information Bottleneck compression with logistic regression classification

**Note**: All models show low recall (~2-3%), indicating difficulty predicting the minority class (defaults). This is common in imbalanced datasets and suggests potential areas for improvement (e.g., class weighting, SMOTE, or threshold tuning).

## 8. Model Saving & Deployment

Save the best performing model (InfoBottleneck-LogReg) and all necessary components for future predictions.

We'll save:
1. **Encoder** (Keras model) as `.h5` file - the Information Bottleneck representation
2. **Autoencoder** (full model) as `.h5` file - for reconstruction tasks
3. **Preprocessor** (sklearn) as `.joblib` file - for data preprocessing
4. **Scaler** (for bottleneck features) as `.joblib` file
5. **Classifier** (Logistic Regression) as `.joblib` file - for final predictions


In [ ]:
# Ensure imports are available (in case first cell wasn't run)
import os
import joblib
import warnings
import logging

# Suppress HDF5 format warnings (we're intentionally using .h5 format)
warnings.filterwarnings('ignore')
logging.getLogger('absl').setLevel(logging.ERROR)

# Create models directory if it doesn't exist
os.makedirs('saved_models', exist_ok=True)

# Save 1: Encoder (Information Bottleneck) as .h5
encoder.save('saved_models/encoder.h5')
print("✅ Saved: saved_models/encoder.h5")

# Save 2: Full Autoencoder as .h5
autoencoder.save('saved_models/autoencoder.h5')
print("✅ Saved: saved_models/autoencoder.h5")

# Save 3: Preprocessor (for raw data → processed features)
joblib.dump(preprocessor, 'saved_models/preprocessor.joblib')
print("✅ Saved: saved_models/preprocessor.joblib")

# Save 4: Scaler for bottleneck features
joblib.dump(scaler_z, 'saved_models/bottleneck_scaler.joblib')
print("✅ Saved: saved_models/bottleneck_scaler.joblib")

# Save 5: Logistic Regression Classifier
joblib.dump(ib_clf, 'saved_models/classifier.joblib')
print("✅ Saved: saved_models/classifier.joblib")

# Save feature names and metadata for reference
metadata = {
    'feature_names': feature_names,
    'selected_feature_names': selected_feature_names.tolist() if 'selected_feature_names' in globals() else None,
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'encoding_dim': encoding_dim,
    'input_dim': input_dim
}
joblib.dump(metadata, 'saved_models/metadata.joblib')
print("✅ Saved: saved_models/metadata.joblib")

print("\n🎉 All model components saved successfully!")


In [ ]:
# Example: How to load and use saved models for predictions

def load_models():
    """Load all saved model components"""
    encoder = tf.keras.models.load_model('saved_models/encoder.h5')
    preprocessor = joblib.load('saved_models/preprocessor.joblib')
    scaler_z = joblib.load('saved_models/bottleneck_scaler.joblib')
    classifier = joblib.load('saved_models/classifier.joblib')
    metadata = joblib.load('saved_models/metadata.joblib')
    return encoder, preprocessor, scaler_z, classifier, metadata

def predict_default(new_data_df, encoder, preprocessor, scaler_z, classifier):
    """
    Predict loan default probability for new borrowers.
    
    Parameters:
    -----------
    new_data_df : pd.DataFrame
        DataFrame with same columns as training data (excluding 'Default' and 'LoanID')
        Must include the 'RiskEntropy' column if it was used in training
    
    Returns:
    --------
    predictions : array
        Binary predictions (0 = no default, 1 = default)
    probabilities : array
        Probability of default for each sample
    """
    # Preprocess the new data
    X_processed = preprocessor.transform(new_data_df)
    
    # Encode to bottleneck representation
    Z = encoder.predict(X_processed, verbose=0)
    
    # Scale bottleneck features
    Z_scaled = scaler_z.transform(Z)
    
    # Predict
    probabilities = classifier.predict_proba(Z_scaled)[:, 1]
    predictions = classifier.predict(Z_scaled)
    
    return predictions, probabilities

# Example usage (commented out - uncomment to test)
# encoder_loaded, preprocessor_loaded, scaler_z_loaded, classifier_loaded, metadata_loaded = load_models()
# 
# # Test on a single row from test set
# sample = X_test.iloc[0:1].copy()
# preds, probs = predict_default(sample, encoder_loaded, preprocessor_loaded, scaler_z_loaded, classifier_loaded)
# print(f"Prediction: {preds[0]} (Default: {preds[0] == 1})")
# print(f"Probability of default: {probs[0]:.4f}")

print("✅ Model loading functions defined. Uncomment the example code to test predictions.")


## 9. Takeaways for Report / PPT / Viva

- **Mutual Information** trimmed the feature space from all engineered columns to the most informative subset without hurting accuracy, proving that low-information dimensions can be safely removed.
- **Shannon Entropy** exposed that defaulters have noticeably higher borrower-level risk entropy, which acts as a novel predictive feature grounded in uncertainty theory.
- **Information Bottleneck** (autoencoder bottleneck layer) yielded a compact latent space that still preserves default signal; the logistic regression on this compressed space performs competitively, showing how deep compression + classification can work for credit risk.
- **Information Gain** from the entropy-based decision tree surfaces interpretable splits (e.g., credit score, DTI bands), linking back to MI rankings and offering explainability for stakeholders.

This end-to-end story connects each Information Theory concept to a tangible modeling step, making the project viva-ready and presentation-friendly.

**Best Model**: InfoBottleneck-LogReg achieves **ROC AUC: 0.7488** by combining Information Bottleneck compression with logistic regression classification.
